In [1]:
import pandas as pd
import geopandas as gpd
import json
import datetime
from collections import defaultdict
import numpy as np
import requests
import re
import os

import sys
sys.path.append('..')
from helpers import (
    get_bulk_power_sales,
    get_county2zone,
    get_service_territories,
    get_dst_timestamps
)

In [2]:
state_abbrev_name_map = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'District of Columbia': 'DC',
    'Delaware': 'DE',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC',
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY'
}

In [3]:
parent_bas = [
    'CISO',
    'ERCO',
    'ISNE',
    'MISO',
    'NYIS',
    'PJM',
    'SWPP',
]

county_subba_maps = {}
for ba in parent_bas:
    with open(f"../config/{ba.lower()}_county_subregion_map.json") as f:
        county_subba_maps[ba] = json.load(f)

utility_subba_maps = {}
for ba in parent_bas:
    try:
        with open(f"../config/{ba.lower()}_subregion_utility_map.json") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"No mapping between sub-BAs and utilities for {ba}")
        continue

    utility_subba_map = {}
    for k, v in data.items():
        if isinstance(v, str):
            utility_subba_map[v] = k
        else:
            for v_ in v:
                utility_subba_map[v_] = k

    utility_subba_maps[ba] = utility_subba_map

No mapping between sub-BAs and utilities for ERCO
No mapping between sub-BAs and utilities for ISNE
No mapping between sub-BAs and utilities for MISO
No mapping between sub-BAs and utilities for NYIS


In [4]:
cplw_counties = [
    'p47029',
    'p37087',
    'p37099',
    'p37115',
    'p37021',
    'p37089',
    'p47171',
    'p37199',
    'p37111',
    'p37161',
    'p37121',
    'p37011'
]

service_territories_by_year = {}
for load_year in range(2016, 2024):
    county2zone = get_county2zone(load_year)
    service_territories = get_service_territories(county2zone, load_year)
    bulk_power_sales = get_bulk_power_sales(load_year, {})
    
    service_territories = (
        service_territories.merge(
            bulk_power_sales[['Utility Number', 'State', 'BA Code']],
            on=['Utility Number', 'State'],
            how='outer'
        )
        .merge(county2zone[['county_state', 'FIPS']], on='county_state', how='left')
    )
    
    service_territories.loc[(
        (service_territories['BA Code'] == 'CPLE')
        & (service_territories.FIPS.isin(cplw_counties))
    ), 'BA Code'] = 'CPLW'
    
    service_territories = service_territories.dropna(subset='BA Code').copy()
    service_territories['eia_codes'] = service_territories['BA Code'].apply(lambda x: [x])
    
    for ba, county_subba_map in county_subba_maps.items():
        service_territories.loc[service_territories['BA Code'] == ba, 'eia_codes'] = (
            service_territories.loc[service_territories['BA Code'] == ba, 'FIPS'].map(county_subba_map)
        )
    
    for ba, subba_utility_map in utility_subba_maps.items():
        service_territories.loc[service_territories['BA Code'] == ba, 'utility_eia_code'] = (
            service_territories.apply(axis=1, func=lambda x: subba_utility_map.get(
                x['Utility Number'], np.nan
            ))
        )
    
    service_territories.loc[service_territories.utility_eia_code.notna(), 'eia_codes'] = (
        service_territories.loc[service_territories.utility_eia_code.notna(), 'utility_eia_code'].apply(lambda x: [x])
    )

    service_territories_by_year[load_year] = service_territories

In [5]:
# Read in demand profiles that have been processed by the EIA data cleaner
# imputation script
regional_profiles = {}
hourly_demand_path = "../eia_data_cleaner/data/regional/outputs"
for fname in os.listdir(hourly_demand_path):
    fpath = os.path.join(hourly_demand_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    region = fname.replace('.csv', '')
    regional_profiles[region] = df

subregional_profiles = {}
hourly_subregion_demand_path = "../eia_data_cleaner/data/subregional/outputs"
for fname in os.listdir(hourly_subregion_demand_path):
    fpath = os.path.join(hourly_subregion_demand_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    subregion = fname.replace('.csv', '')
    subregional_profiles[subregion] = df

regions = list(regional_profiles.keys())
subregions = list(subregional_profiles.keys())

In [6]:
# Create name mappings for EIA-930 respondents, subregions, etc.
hourly_rto_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_rto.csv"
)
respondent_name_map = dict(zip(
    hourly_rto_demand['respondent'],
    hourly_rto_demand['respondent-name']
))

hourly_subregion_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_subregion.csv",
    dtype={'subba': str}
)
subba_name_map = dict(zip(
    hourly_subregion_demand['subba'], hourly_subregion_demand['subba-name']
))
parent_name_map = dict(zip(
    hourly_subregion_demand['parent'], hourly_subregion_demand['parent-name']
))

In [7]:
with open('../config/epm_to_861_utility_map.json', 'r') as f:
    epm_to_861_utility_map = json.load(f)

with open('../config/epm_to_861_ba_map.json', 'r') as f:
    epm_to_861_ba_map = json.load(f)

with open('../config/epm_respondent_timezone_map.json', 'r') as f:
    epm_respondent_timezone_map = json.load(f)

In [8]:
def validate_states_and_counties(state, counties, service_territories, print_corrections=False):
    state_territories = (
        service_territories.loc[(
            service_territories.State == state
        )]
        .dropna(subset='FIPS')
        [['State', 'County', 'FIPS']]
        .drop_duplicates()
    )
    valid_counties = state_territories.County.unique().tolist()
    invalid_counties = [county for county in counties if county not in valid_counties]
    for county in invalid_counties:
        if print_corrections:
            print(f"Removing {county} from event in {state}")
        counties.remove(county)

        updated_county = county.rsplit(' ', maxsplit=1)[0]
        if updated_county not in valid_counties:
            potential_counties = [county for county in valid_counties if (
                (county in valid_counties) and (county not in counties) and (county.startswith(updated_county))
            )]

            if len(potential_counties) == 0:
                if print_corrections:
                    print(f"No valid counties represent {updated_county} in {state}")
                continue

            assert len(potential_counties) == 1, f"More than one potential county for {updated_county} in {state}"
            updated_county = potential_counties[0]

        if print_corrections:
            print(f"Adding {updated_county} to event in {state}")
        counties.append(updated_county)

    return counties

def get_affected_states_and_counties(affected_area, service_territories):
    affected_area = (
        affected_area.replace('Parish', 'County')
        .replace(', City of', ' City')
        .replace('County:', 'County,')
        .replace('St.', 'St')
        .replace(': City and County of', ' County')
    )
    
    state_county_map = {}

    if ':' not in affected_area:
        for area in affected_area.split(','):
            state = state_abbrev_name_map[area.lstrip()]
            state_county_map[state] = []
    else:
        all_areas = sum([area.split(':') for area in affected_area.split(';') if area != ''], [])
        for i in range(len(all_areas) - 1):
            area = all_areas[i].lstrip()
            if 'County' in area:
                continue
        
            state = state_abbrev_name_map[area.lstrip()]
            if 'County' in (next_area := all_areas[i+1]):
                if state == 'LA':
                    next_area = (
                        next_area.replace('De Soto', 'DeSoto')
                        .replace('LaSalle', 'La Salle')
                    )
                counties = [
                    county.replace(' County', '').lstrip()
                    for county in next_area.split(',')
                    if county != ''
                ]
                state_county_map[state] = validate_states_and_counties(
                    state, counties, service_territories
                )
            else:
                state_county_map[state] = []

    return state_county_map

In [9]:
dropped_bas = [
    'AVRN',
    'DEAA',
    'EEI',
    'GLHB',
    'GRID',
    'GRIF',
    'GRMA',
    'GWA',
    'HGMA',
    'SEPA',
    'WWA',
    'YAD',
    'OVEC',
    'SEC',
    'NBSO'
]

In [10]:
def get_eia_codes(event, service_territories, print_missing_areas=False):
    utility_or_power_pool = event["Utility/Power Pool"]        
    affected_area = event["Area Affected"]
    affected_states_and_counties = get_affected_states_and_counties(affected_area, service_territories)
    
    if ba_codes := epm_to_861_ba_map.get(utility_or_power_pool):
        if (len(ba_codes) == 1) and (ba_codes[0] not in parent_bas):
            return ba_codes, ba_codes
        else:
            affected_territories = service_territories.loc[service_territories['BA Code'].isin(ba_codes)]
    elif utility_ids := epm_to_861_utility_map.get(utility_or_power_pool):
        affected_territories = service_territories.loc[service_territories['Utility Number'].isin(utility_ids)]
    else:
        print(f"No mapping for {utility_or_power_pool}")
        return [], []

    affected_territories = affected_territories.loc[(
        affected_territories['State'].isin(affected_states_and_counties.keys())
    )]
    for state, counties in affected_states_and_counties.items():
        if len(counties) > 0:
            affected_territories = affected_territories.loc[(
                (affected_territories['State'] != state)
                | (affected_territories['County'].isin(counties))
            )]

    if print_missing_areas:
        for state, counties in affected_states_and_counties.items():
            if len(counties) > 0:
                affected_territory_counties = affected_territories.loc[(
                    (affected_territories.State == state) & (affected_territories.County.isin(counties))
                ), 'County'].unique().tolist()
                counties = list(set(counties))
                if len(affected_territory_counties) != len(counties):
                    print('')
                    print(f"Missing {[county for county in counties if (county not in affected_territory_counties)]} for {event['Utility/Power Pool']} ({event.name})")
                    print('')
            else:
                if state not in affected_territories.State.unique().tolist():
                    print('')
                    print(f"Missing {state} for {event['Utility/Power Pool']} ({event.name})")
                    print('')
    
    ba_codes = list(affected_territories.dropna(subset='BA Code')['BA Code'].unique())
    eia_codes = list(set(sum(affected_territories.dropna(subset='eia_codes')['eia_codes'].tolist(), [])))

    return ba_codes, eia_codes

In [11]:
timezone_to_utc_shift_map = {
    "Eastern": 5,
    "Central": 6,
    "Arizona": 7,
    "Mountain": 7,
    "Pacific": 8,
}

def add_eia_codes(df, year):
    service_territories = service_territories_by_year[year]
    all_ba_codes = []
    all_eia_codes = []
    
    for i, row in df.iterrows():
        utility = row['Utility/Power Pool']
        affected_area = row['Area Affected']
        ba_codes, eia_codes = get_eia_codes(row, service_territories)
        all_ba_codes.append(ba_codes)
        all_eia_codes.append(eia_codes)

    df['ba_codes'] = all_ba_codes
    df['eia_codes'] = all_eia_codes

    return df

def clean_start_end_times(df, year):
    df['start_time'] = pd.to_datetime(df['Event Date and Time'], format='mixed')
    df['end_time'] = pd.to_datetime(df['Restoration Date and Time'], format='mixed')
    
    dst_start, dst_end = get_dst_timestamps(year)
    
    dst_start_time_mask = (df['start_time'] >= dst_start) & (df['start_time'] <= dst_end)
    df.loc[dst_start_time_mask, 'start_time'] -= pd.Timedelta(hours=1)
    
    dst_end_time_mask = (df['end_time'] >= dst_start) & (df['end_time'] <= dst_end)
    df.loc[dst_end_time_mask, 'end_time'] -= pd.Timedelta(hours=1)

    df['start_time'] = (
        df.apply(
            axis=1,
            func=lambda x: x['start_time'] + pd.Timedelta(hours=x['timezone_shift'] + 1)
        )
        .dt
        .floor('h')
    )
    df['end_time'] = (
        df.apply(
            axis=1,
            func=lambda x: x['end_time'] + pd.Timedelta(hours=x['timezone_shift'] + 1)
        )
        .dt
        .floor('h')
    )

    return df

In [12]:
dropped_utilities = [
    'Seminole Electric Cooperative Inc',
    'Seminole Electric Cooperative Inc.'
]

def get_load_loss_events(load_year):
    df = pd.read_excel(f'../data/disturbance_events/Table_B_2_{load_year}.xlsx')
    df.columns = df.loc[0]
    df = (
        df.loc[1:]
        .dropna(subset='Restoration Date and Time')
        .reset_index(drop=True)
    )
    df = (
        df.loc[(
            (~df['Restoration Date and Time'].str.contains('\\.'))
            & (~df['Event Date and Time'].str.contains('\\.'))
            & (~df['Utility/Power Pool'].isin(dropped_utilities))
        )]
        .copy()
        .reset_index(drop=True)
    )

    df['Area Affected'] = (
        df['Area Affected'].apply(lambda x: re.sub(r'[^a-zA-Z,:;. ]', '', x))
    )
    
    df = (
        df.loc[(
            ~df['Area Affected'].isin(['Puerto Rico', 'Puerto Rico:'])
        )]
        .copy()
        .reset_index(drop=True)
    )

    df = add_eia_codes(df, load_year)
    df['timezone_shift'] = (
        df['Utility/Power Pool']
        .map(epm_respondent_timezone_map)
        .map(timezone_to_utc_shift_map)
    )
    df = df.dropna(subset='timezone_shift')
    df = clean_start_end_times(df, load_year)

    return df

In [13]:
load_loss_events_by_year = {}
for year in range(2016, 2024):
    print(year)
    load_loss_events_by_year[year] = get_load_loss_events(year)

load_loss_events = pd.concat(load_loss_events_by_year, ignore_index=True)

2016
No mapping for California Department of Water Resources
No mapping for California Department of Water Resources
No mapping for Upstate New York Power Producers
No mapping for CAmbria Cogen Company
No mapping for Peak Reliability
No mapping for Broad River Energy, LLC
No mapping for Peak Reliability
No mapping for Peak Reliability
No mapping for Peak Reliability
No mapping for Upstate New York Power Producers
No mapping for Peak Reliability
No mapping for Peak Reliability
No mapping for California Department of Water Resources
2017
No mapping for North Carolina Mun Power Agny #1
No mapping for Western Area Power Administration - Western Area Lower Colorado
No mapping for California Department of Water Resources
No mapping for Peak Reliability
No mapping for Peak Reliability
No mapping for Peak Reliability
2018
No mapping for Somerset Operating Company, LLC
No mapping for Louisiana Generating LLC
No mapping for Peak Reliability
No mapping for Louisiana Generating LLC
2019
No mapping

In [14]:
load_loss_events.to_csv("../data/load_loss_events.csv", index=False)